# 공개 로봇 데이터와 VLA 온보딩 · 실습


## 학습 목표

<!-- exercise-guidance:reinforcement-learning/week-10/objective -->
공개 시연 데이터를 직접 읽고 평가 기준을 만든 뒤, 시각·언어·행동 모델에 연결할 입력과 출력의 의미를 확인합니다.
<!-- /exercise-guidance:reinforcement-learning/week-10/objective -->


## 실행 조건

<!-- exercise-guidance:reinforcement-learning/week-10/execution -->
작은 데이터 분석에는 requests·pandas·pyarrow·scikit-learn이 필요합니다. 공개 Parquet 한 개만 읽으며 인터넷 연결이 필요합니다. VLA 모델 추론·학습은 별도의 장비와 모델별 환경을 사용합니다.

필요한 패키지가 없으면 Colab·Jupyter 코드 셀에서 `%pip install pyarrow==23.0.1`를 실행하세요.
<!-- /exercise-guidance:reinforcement-learning/week-10/execution -->


## 적용 사례

<!-- exercise-guidance:reinforcement-learning/week-10/case-studies -->
### 시연 데이터에서 Pick-and-place 정책으로

공개 집기·이동 시연을 읽고 행동의 의미와 평가 분리를 확인합니다. 작은 오프라인 기준 모델에서 시각·언어를 사용하는 정책으로 확장합니다.

**관측·상태:** 동기화된 카메라 영상·로봇 상태·작업 지시. Dataset의 Feature·시간 기준을 확인.

**행동:** Dataset에서 정의한 관절·Gripper 등의 Action과 Action chunk. 표현·정규화를 먼저 대응.

**보상·제약:** 시연학습의 예측 오차와 실제 Task 성공을 분리. 관절·입력 범위와 실패 조건 기록.

**비교 기준:** state-only Ridge의 오프라인 행동 예측. 폐루프 정책 평가는 별도 수행.

**확인할 지표:** Episode 분리의 성분별 MAE. 실제·시뮬레이션 실행 시 성공률·안전 위반·지연 추가.
<!-- /exercise-guidance:reinforcement-learning/week-10/case-studies -->


## 공개 로봇 Episode 읽기

<!-- exercise-guidance:reinforcement-learning/week-10/activity-1 -->
### 시작 조건

LeRobot svla_so100_pickplace의 첫 Parquet shard를 읽습니다. 영상·모델 가중치는 내려받지 않습니다.

### 진행 방법

1. 제공된 코드로 Repository revision·FPS·Episode 수·state/action shape를 확인하세요.
2. Episode 0의 state와 action을 시간축으로 그려, 두 배열의 성분·단위를 데이터 카드와 대조하세요.
3. 카메라·로봇 상태·자연어 지시가 어느 metadata와 Feature로 연결되는지 적으세요.

### 남길 결과

- 데이터 구조 표와 Episode 0의 state/action 그림.
<!-- /exercise-guidance:reinforcement-learning/week-10/activity-1 -->


In [ ]:
"""Read one public LeRobot data shard, without downloading videos or model weights."""
import io,json
import numpy as np
import pandas as pd
import requests
repo='lerobot/svla_so100_pickplace'
def read_json(url):
    response=requests.get(url,timeout=30);response.raise_for_status();return response.json()
manifest=read_json('https://huggingface.co/api/datasets/'+repo)
revision=manifest['sha']
base='https://huggingface.co/datasets/'+repo+'/resolve/'+revision+'/'
metadata=read_json(base+'meta/info.json')
shards=sorted(x['rfilename'] for x in manifest['siblings'] if x['rfilename'].startswith('data/') and x['rfilename'].endswith('.parquet'))
assert shards,'Dataset layout changed; inspect the repository data tree.'
response=requests.get(base+shards[0],timeout=60);response.raise_for_status()
frame=pd.read_parquet(io.BytesIO(response.content))
print('Repository / revision / shard:',repo,revision,shards[0])
print('Format / FPS / episodes:',metadata.get('codebase_version'),metadata['fps'],metadata['total_episodes'])
print('Features:',metadata['features'])
print('Loaded rows / episodes:',len(frame),frame['episode_index'].nunique())
state=np.stack(frame['observation.state'].to_numpy())
action=np.stack(frame['action'].to_numpy())
print('State / action shape:',state.shape,action.shape)
# Task text and video paths are described in the repository metadata.
# No robot commands, model execution, remote uploads, or video downloads occur here.



## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-10/activity-1-record -->
**항목:**

**Feature·metadata:**

**shape·단위:**

**확인한 의미:**

### 확인할 점

- 관절 목표와 현재 관절 상태를 같은 값으로 가정하지 않았는가?
- Dataset revision과 이용 조건을 확인했는가?
<!-- /exercise-guidance:reinforcement-learning/week-10/activity-1-record -->


## Episode 단위 평가와 정보 누출

<!-- exercise-guidance:reinforcement-learning/week-10/activity-2 -->
### 시작 조건

로드한 Episode를 앞 80%·뒤 20%로 나눕니다. 상태에서 행동을 예측하는 Ridge 기준 모델을 만듭니다.

### 진행 방법

1. 시간적으로 인접한 Frame을 임의 분할하는 것과 Episode 분할의 차이를 적으세요.
2. 기준 모델을 학습하고 각 Action 성분의 MAE를 Episode 평가 자료에서 계산하세요.
3. 같은 자료의 무작위 Frame 분할과 비교해 수치가 좋아 보이는 이유를 설명하세요.

### 남길 결과

- 분할별 오차 표와 Action 한 성분의 예측 그림.
<!-- /exercise-guidance:reinforcement-learning/week-10/activity-2 -->


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
episodes=np.sort(frame['episode_index'].unique());cut=max(1,int(.8*len(episodes)))
train=np.isin(frame['episode_index'],episodes[:cut]);test=~train
assert test.any(),'Select a shard containing at least two episodes.'
model=Ridge(alpha=1.).fit(state[train],action[train])
predicted=model.predict(state[test])
print('Episode split MAE:',mean_absolute_error(action[test],predicted,multioutput='raw_values'))
# TODO: 같은 자료의 Frame 분할과 비교하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-10/activity-2-record -->
**분할:**

**학습 Episode 수:**

**평가 Episode 수:**

**성분별 MAE:**

**누출 가능성:**

### 확인할 점

- 전처리·회귀 fit에 평가 Episode를 사용하지 않았는가?
- 기록 Action 예측 오차를 로봇 Task 성공률이라고 부르지 않았는가?
<!-- /exercise-guidance:reinforcement-learning/week-10/activity-2-record -->


## VLA로 확장할 때 필요한 입력

<!-- exercise-guidance:reinforcement-learning/week-10/activity-3 -->
### 시작 조건

SmolVLA 공식 설명과 현재 Dataset의 Feature를 사용합니다. 모델 전체 학습 대신 입출력·평가 설계를 완성합니다.

### 진행 방법

1. 카메라·현재 상태·지시문→Action chunk의 대응을 표로 적으세요.
2. Action 정규화·좌표계·제어 주기·Chunk 실행 중 지연을 점검할 위치를 정하세요.
3. 자신의 연구에 필요한 추가 관측·지시·평가 Episode 한 가지를 설계하세요. GPU가 준비되면 공식 예제로 사전학습 정책 추론을 확장합니다.

### 남길 결과

- VLA 입출력 표와 연구 적용 계획.
<!-- /exercise-guidance:reinforcement-learning/week-10/activity-3 -->


## 결과 기록·자가 점검

<!-- exercise-guidance:reinforcement-learning/week-10/activity-3-record -->
**입력·출력:**

**Dataset 대응:**

**정규화·시간 기준:**

**평가 방법:**

### 확인할 점

- VLA의 행동 출력과 VLM의 장면 설명을 구분했는가?
- 사전학습 모델이 다른 로봇·Action 표현에 즉시 맞는다고 가정하지 않았는가?
<!-- /exercise-guidance:reinforcement-learning/week-10/activity-3-record -->


## 더 해보기

<!-- exercise-guidance:reinforcement-learning/week-10/optional -->
VLM을 상위 목표 해석기로 사용할 경우, vLLM의 Multimodal 입력 예제로 이미지·지시→구조화된 목표를 시험하세요. 추론 지연·출력 유효성을 측정하고, 저수준 로봇 Action과 연결하기 전 좌표·범위 검증을 분리하세요.
<!-- /exercise-guidance:reinforcement-learning/week-10/optional -->


## 공식 예제·데이터

<!-- exercise-guidance:reinforcement-learning/week-10/resources -->
- [SmolVLA · 구조와 사용법](https://huggingface.co/docs/lerobot/smolvla)
- [공개 시연 Dataset](https://huggingface.co/datasets/lerobot/svla_so100_pickplace)
- [Episode 시각화](https://huggingface.co/spaces/lerobot/visualize_dataset?path=/lerobot/svla_so100_pickplace/episode_0)
- [vLLM · Multimodal 입력](https://docs.vllm.ai/en/latest/features/multimodal_inputs/)
<!-- /exercise-guidance:reinforcement-learning/week-10/resources -->
